# 🎯 [K07_M02] 공지 문장 나누고 정리하기
### 학생 실습 워크북 & 파이썬 실습 가이드 (Google Colab / Jupyter)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/K07/k07_m02_student.ipynb)

---

## 📌 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 / 과제** | 일반행정·부대관리 (Level 2) / K07 군 공지 자동 요약 |
| **미션** | 미션 02. 공지 문장 나누고 정리하기 (소요시간: 약 2.0시간) |
| **실습 유형** | 파이썬 자연어 처리(NLP) 및 텍스트 자동 요약 파이프라인 구축 |

> **⚠️ [교육용 가상 자료 원칙]**
> 본 실습의 모든 자료는 안전 규정에 따라 **교육용 가상 부대 자료(가상 한빛대대)**로 구성되어 있습니다.
> 실제 군사 기밀이나 부대 세부 인적·작전 정보는 외부 AI 도구에 절대 입력하지 않습니다.



In [1]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 02. 공지 문장 나누고 정리하기
**소요시간**: 약 2.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

실제 부대 실무 현장에서는 불명확한 정보 전달이나 수동 업무 처리로 인해 불필요한 행정 소요가 발생하고 주요 공지·규정 확인이 지연되는 문제가 발생하기 쉽습니다. 따라서 **공지 본문을 문장 단위로 나누고 요약하기 좋은 형태로 정리합니다.** 과업을 체계적으로 수행하여 현장 업무 효율성을 극대화하고 정보 누락으로 인한 혼선을 사전에 방지해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **마침표를 기준으로 문장을 나누되 숫자 사이 마침표(예: 2.5)는 나누지 않도록 규칙을 만들고, 공백과 괄호 표기를 정리하고 체계화하는** 실습을 단계별로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동을 순차적으로 진행하며 핵심 산출물을 도출하고 검증 기준을 달성합니다.

**이번 미션 결과물:** `문장 분리·정리 함수, 공지별 문장 수 표` 완성

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가 기준:** [필수] 전처리 — 예외를 고려한 문장 분리 규칙을 만들고 결과를 확인했는가? (PASS / FAIL)

#### 미션 PASS 조건

- 요구 산출물(문장 분리·정리 함수, 공지별 문장 수 표)의 완성도 충족


- 원문 대조 점검을 통한 사실 검증 및 정확성 확보


- 교육용 가상 데이터 활용 및 군사보안 수칙 엄수


#### 채점자 판정 메모

> 단순 결과물 생성 여부뿐만 아니라 AI 결과물의 오류를 직접 찾아내고 바로잡았는지 대조표를 확인한다.



In [ ]:
"""공지 본문을 문장 단위로 나누고, 공백·괄호 표기를 정리해 요약하기 좋은 형태로 바꾼다."""
import re
import pandas as pd

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 한빛대대 자료 (실제 부대·규정·인물과 무관)
# ---------------------------------------------------------------------------
NOTICES = [{'id': 'N01',
  'date': '2026-11-02',
  'from': '대대 행정과',
  'title': '11월 정기 휴가 신청 접수 안내',
  'body': '11월 정기 휴가 신청을 11월 5일(목) 18시까지 접수합니다. 신청은 중대 행정반에 비치된 휴가 신청서를 작성하여 소속 분대장에게 제출합니다. 휴가 기간이 겹치는 인원이 '
          '중대 정원의 20%를 넘으면 선착순이 아닌 추첨으로 조정합니다. 결과는 11월 9일(월) 중대 게시판에 게시합니다. 문의: 행정과 담당관(내선 1101).'},
 {'id': 'N02',
  'date': '2026-11-02',
  'from': '대대 의무실',
  'title': '독감 예방접종 일정 안내',
  'body': '독감 예방접종을 11월 10일(화)부터 11월 12일(목)까지 중대별로 실시합니다. 1중대는 10일 09시, 2중대는 11일 09시, 3중대는 12일 09시에 의무실 앞으로 '
          '집합합니다. 접종 당일 발열이 있는 인원은 접종 전 의무병에게 알리고, 접종 후 30분간 의무실 대기 후 복귀합니다. 미접종자는 11월 16일(월) 추가 접종을 받습니다.'},
 {'id': 'N03',
  'date': '2026-11-03',
  'from': '대대 군수과',
  'title': '동계 피복 보급 일정',
  'body': '동계 피복(방한 내피, 방한 장갑, 넥워머)을 11월 6일(금) 13시부터 17시까지 중대 순서대로 보급합니다. 분대장은 분대원 치수표를 11월 4일(수)까지 군수과에 '
          '제출합니다. 치수 교환은 보급 당일에만 가능하며, 이후에는 다음 달 정기 보급 때 교환합니다.'},
 {'id': 'N04',
  'date': '2026-11-03',
  'from': '대대 정보통신실',
  'title': '사이버 보안 점검 및 비밀번호 변경',
  'body': '11월 9일(월)부터 13일(금)까지 사이버 보안 점검을 실시합니다. 모든 인원은 11월 8일(일)까지 업무용 계정 비밀번호를 변경해야 합니다. 비밀번호는 '
          '영문·숫자·특수문자를 포함해 9자리 이상으로 설정하고, 이전 비밀번호와 같으면 안 됩니다. 개인 휴대전화로 업무 자료를 촬영하거나 전송하는 행위는 금지됩니다.'},
 {'id': 'N05',
  'date': '2026-11-04',
  'from': '대대 교육훈련과',
  'title': '11월 사격 훈련 일정 변경',
  'body': '기상 악화 예보로 11월 6일(금) 예정이던 2중대 사격 훈련을 11월 13일(금)로 연기합니다. 1중대와 3중대 일정은 변경 없습니다. 2중대는 6일에 실내 교육(사격술 '
          '예비훈련)으로 대체하며, 장소는 대대 강당입니다. 세부 시간표는 중대장 지시에 따릅니다.'},
 {'id': 'N06',
  'date': '2026-11-04',
  'from': '대대 복지과',
  'title': '부대 내 복지시설 운영 시간 조정',
  'body': '동절기를 맞아 11월 9일(월)부터 체력단련실 운영 시간을 06시~21시로, 사이버지식정보방 운영 시간을 18시~21시 30분으로 조정합니다. 매주 수요일 14시~16시는 '
          '체력단련실 소독으로 이용할 수 없습니다. 복지매장(PX) 운영 시간은 기존과 같습니다.'},
 {'id': 'N07',
  'date': '2026-11-05',
  'from': '대대 인사과',
  'title': '자기개발 지원 신청 안내',
  'body': '2027년 상반기 자기개발 비용 지원 신청을 11월 20일(금)까지 받습니다. 지원 대상은 자격증 응시료와 온라인 강좌 수강료이며, 1인당 연간 한도 내에서 실비를 '
          '지원합니다. 신청 시 응시 확인서 또는 수강 등록 확인서를 첨부합니다. 지원 여부는 12월 첫째 주에 개별 통보합니다.'},
 {'id': 'N08',
  'date': '2026-11-05',
  'from': '대대 행정과',
  'title': '11월 정기 휴가 신청 접수 기한 연장',
  'body': '11월 정기 휴가 신청 접수 기한을 11월 5일(목) 18시에서 11월 6일(금) 12시로 연장합니다. 사격 훈련 일정 변경에 따른 조치입니다. 결과 게시일(11월 9일)은 '
          '변경 없습니다.'},
 {'id': 'N09',
  'date': '2026-11-06',
  'from': '대대 안전관리관',
  'title': '동절기 안전사고 예방 수칙',
  'body': '동절기 안전사고 예방을 위해 다음 사항을 지킵니다. 생활관 내 개인 전열기 사용을 금지합니다. 차량 운행 전 타이어 상태와 결빙 구간을 확인합니다. 야외 활동 시 방한 장구를 '
          '착용하고 2인 1조로 움직입니다. 동상 의심 증상이 있으면 즉시 의무실에 알립니다.'},
 {'id': 'N10',
  'date': '2026-11-06',
  'from': '대대 의무실',
  'title': '독감 예방접종 3중대 일정 변경',
  'body': '3중대 독감 예방접종을 11월 12일(목) 09시에서 11월 12일(목) 14시로 변경합니다. 오전 훈련 일정 조정에 따른 것입니다. 1중대와 2중대 일정은 변경 없습니다.'},
 {'id': 'N11',
  'date': '2026-11-09',
  'from': '대대 군수과',
  'title': '생활관 난방기 점검',
  'body': '11월 11일(수)부터 13일(금)까지 생활관 난방기를 점검합니다. 점검 시간에는 생활관을 비워 두고, 귀중품은 개인 관물대에 보관합니다. 난방기 이상 소음이나 냄새가 있으면 '
          '점검 전까지 중대 행정반에 신고합니다.'},
 {'id': 'N12',
  'date': '2026-11-09',
  'from': '대대 교육훈련과',
  'title': 'AI 활용 교육 참가자 모집',
  'body': '대대 자체 AI 활용 교육 참가자를 모집합니다. 교육은 11월 23일(월)부터 4주간 매주 월요일 19시~21시에 사이버지식정보방에서 진행합니다. 중대별 5명 이내로 11월 '
          '16일(월)까지 중대 행정반에 신청합니다. 부대 자료를 외부 AI 서비스에 입력하지 않는 보안 수칙 교육이 포함됩니다.'}]


def split_sentences(text):
    # [TODO : 학생 실습 2] 공지 문장 나누고 정리하기 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def normalize(s):
    # [TODO : 학생 실습 2] 공지 문장 나누고 정리하기 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
rows = []
for n in NOTICES:
    sents = [normalize(s) for s in split_sentences(n["body"])]
    rows.append({"id": n["id"], "제목": n["title"], "문장 수": len(sents), "첫 문장": sents[0][:40] + "..."})
df = pd.DataFrame(rows)
print(df.to_string(index=False))
print(f"\n전체 문장 {df['문장 수'].sum()}개, 공지당 평균 {df['문장 수'].mean():.1f}문장")

print("\n[N01 문장 분리 예시]")
for i, s in enumerate(split_sentences(NOTICES[0]["body"]), 1):
    print(f" {i}. {s}")

# 문장 분리 규칙 점검: 숫자 사이 마침표(예: 2.5)는 나누지 않아야 한다
test = "사격 점수는 2.5점 이상입니다. 결과는 게시합니다."
print("\n규칙 점검:", split_sentences(test))
